In [1]:
import os
import gc
import time
import random
import warnings
import multiprocessing
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

warnings.filterwarnings("ignore")

In [2]:
# ==========================================================
# REPRODUCIBILITY
# ==========================================================

SEED = 42


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

print(f"Random seed set successfully: {SEED}")

Random seed set successfully: 42


In [3]:
# ==========================================================
# DEVICE
# ==========================================================

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DEVICE_TYPE = "cuda"

    GPU_NAME = torch.cuda.get_device_name(0)

    print("Using NVIDIA CUDA GPU")
    print(f"GPU: {GPU_NAME}")

elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    DEVICE_TYPE = "mps"
    GPU_NAME = None

    print("Using Apple MPS")

else:
    DEVICE = torch.device("cpu")
    DEVICE_TYPE = "cpu"
    GPU_NAME = None

    print("Using CPU")

Using NVIDIA CUDA GPU
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [4]:
# ==========================================================
# ENVIRONMENT INFORMATION
# ==========================================================

print("=" * 60)
print("ENVIRONMENT INFORMATION")
print("=" * 60)

print("PyTorch Version :", torch.__version__)
print("Python Version  :", platform.python_version())
print("Operating System:", platform.system(), platform.release())
print("CPU Cores       :", multiprocessing.cpu_count())
print("CUDA Available  :", torch.cuda.is_available())
print("Device          :", DEVICE)

if torch.cuda.is_available():
    total_memory_gb = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print("GPU             :", GPU_NAME)
    print(f"GPU Memory      : {total_memory_gb:.2f} GB")

print("=" * 60)

ENVIRONMENT INFORMATION
PyTorch Version : 2.2.2+cu121
Python Version  : 3.11.9
Operating System: Windows 10
CPU Cores       : 12
CUDA Available  : True
Device          : cuda
GPU             : NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU Memory      : 6.00 GB


In [5]:
# ==========================================================
# PROJECT CONFIGURATION
# ==========================================================

PROJECT_DIR = Path(
    r"D:\Emotion Detection system V2"
).resolve()


# ==========================================================
# CANONICAL EMOTION LABELS
# ==========================================================

EMOTION_LABELS = [
    "Anger",
    "Disgust",
    "Fear",
    "Happiness",
    "Sadness",
    "Surprise",
    "Neutral",
]

LABEL2IDX = {label: idx for idx, label in enumerate(EMOTION_LABELS)}
IDX2LABEL = {idx: label for label, idx in LABEL2IDX.items()}
NUM_CLASSES = len(EMOTION_LABELS)


# ==========================================================
# LABEL NORMALIZATION
# ==========================================================

LABEL_NORMALIZATION = {
    "Angry": "Anger",
    "Anger": "Anger",
    "Disgust": "Disgust",
    "Fear": "Fear",
    "Happy": "Happiness",
    "Happiness": "Happiness",
    "Sad": "Sadness",
    "Sadness": "Sadness",
    "Surprise": "Surprise",
    "Neutral": "Neutral",
}


# ==========================================================
# MODALITIES
# ==========================================================

MODALITIES = [
    "audio",
    "face",
    "text",
    "video",
]

MODALITY2IDX = {
    modality: idx
    for idx, modality in enumerate(MODALITIES)
}


# ==========================================================
# TASKS
# ==========================================================

TASKS = [
    "emotion",
    "sarcasm",
]


# ==========================================================
# ACTIVE DATASET PATHS
# ==========================================================

# These datasets are active in the current project.
# SAMM is archived/optional; GoEmotions is ACTIVE for text emotion.
DATASET_PATHS = {
    # Emotion datasets
    "CREMA_D":
        PROJECT_DIR / "datasets" / "AudioWAV",

    "RAVDESS":
        PROJECT_DIR / "datasets" / "RAVDESS_Dataset",

    "SAVEE":
        PROJECT_DIR
        / "datasets" / "ejlok1"
        / "surrey-audiovisual-expressed-emotion-savee",

    "TESS":
        PROJECT_DIR
        / "datasets" / "ejlok1"
        / "toronto-emotional-speech-set-tess"
        / "versions" / "1"
        / "TESS Toronto emotional speech set data"
        / "TESS Toronto emotional speech set data",

    "IEMOCAP":
        PROJECT_DIR
        / "datasets" / "dejolilandry"
        / "iemocapfullrelease"
        / "versions" / "1"
        / "IEMOCAP_fullrelease",

    "FER2013":
        PROJECT_DIR / "datasets" / "msambare" / "fer2013",

    "AffectNet":
        PROJECT_DIR / "datasets" / "mstjebashazida" / "affectnet",

    "CK_PLUS":
        PROJECT_DIR / "datasets" / "shawon10" / "ckplus",

    "RAF_DB":
        PROJECT_DIR / "datasets" / "shuvoalok" / "raf-db-dataset",

    # Separate sarcasm datasets
    "MUStARD":
        PROJECT_DIR / "datasets" / "MUStARD",

    "NEWS_HEADLINES_SARCASM":
        PROJECT_DIR
        / "datasets" / "rmisra"
        / "news-headlines-dataset-for-sarcasm-detection",
    "GoEmotions": PROJECT_DIR / "datasets" / "debarshichanda" / "goemotions" / "versions" / "6" / "data" / "full_dataset",
}


ACTIVE_EMOTION_DATASETS = [
    "CREMA_D",
    "RAVDESS",
    "SAVEE",
    "TESS",
    "IEMOCAP",
    "FER2013",
    "AffectNet",
    "CK_PLUS",
    "RAF_DB",
    "GoEmotions",
]

ACTIVE_SARCASM_DATASETS = [
    "MUStARD",
    "NEWS_HEADLINES_SARCASM",
]

ARCHIVED_DATASETS = [
    "SAMM",
]


# ==========================================================
# DATASET MODALITY EXPECTATIONS
# ==========================================================

DATASET_MODALITIES = {
    "CREMA_D": ["audio"],
    "RAVDESS": ["audio", "video"],
    "SAVEE": ["audio"],
    "TESS": ["audio"],
    "IEMOCAP": ["audio", "video"],
    "FER2013": ["face"],
    "AffectNet": ["face"],
    "CK_PLUS": ["face"],
    "RAF_DB": ["face"],
    "MUStARD": ["text", "video"],
    "NEWS_HEADLINES_SARCASM": ["text"],
    "GoEmotions": ["text"],
}


# ==========================================================
# FILE-TYPE ROUTING
# ==========================================================

AUDIO_EXTENSIONS = {
    ".wav", ".mp3", ".flac", ".m4a", ".aac", ".ogg"
}

VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv", ".webm"
}

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"
}

TEXT_EXTENSIONS = {
    ".csv", ".txt", ".json", ".jsonl"
}

SUPPORTED_EXTENSIONS = (
    AUDIO_EXTENSIONS
    | VIDEO_EXTENSIONS
    | IMAGE_EXTENSIONS
    | TEXT_EXTENSIONS
)


print("Project directory :", PROJECT_DIR)
print("Emotion classes   :", EMOTION_LABELS)
print("Number of classes :", NUM_CLASSES)
print("Modalities        :", MODALITIES)
print("Tasks             :", TASKS)
print("Active emotion    :", ACTIVE_EMOTION_DATASETS)
print("Active sarcasm    :", ACTIVE_SARCASM_DATASETS)
print("Archived          :", ARCHIVED_DATASETS)


Project directory : D:\Emotion Detection system V2
Emotion classes   : ['Anger', 'Disgust', 'Fear', 'Happiness', 'Sadness', 'Surprise', 'Neutral']
Number of classes : 7
Modalities        : ['audio', 'face', 'text', 'video']
Tasks             : ['emotion', 'sarcasm']
Active emotion    : ['CREMA_D', 'RAVDESS', 'SAVEE', 'TESS', 'IEMOCAP', 'FER2013', 'AffectNet', 'CK_PLUS', 'RAF_DB', 'GoEmotions']
Active sarcasm    : ['MUStARD', 'NEWS_HEADLINES_SARCASM']
Archived          : ['SAMM']


In [6]:
# ==========================================================
# AUTHORITATIVE DATASET RESOLUTION + MODALITY INVENTORY
# ==========================================================
# This is the ONLY raw-dataset inventory cell in the notebook.
# It replaces the previous extension-only inventory.
#
# Important:
# - Physical image files are logically FACE.
# - CSV/TXT/JSON/JSONL inside media datasets are metadata, NOT text samples.
# - GoEmotions is a row-based TEXT dataset: file count != sample count.
# - RAVDESS and IEMOCAP may provide both AUDIO and VIDEO.
# - SAMM remains archived/optional.

from pathlib import Path

IMAGE_EXTS = {".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp"}
AUDIO_EXTS = {".wav",".mp3",".flac",".ogg",".m4a",".aac"}
VIDEO_EXTS = {".mp4",".avi",".mov",".mkv",".webm",".wmv",".flv"}
TEXT_EXTS = {".csv",".txt",".json",".jsonl"}

DATASET_MODALITIES.update({
    "CREMA_D": ["audio"],
    "RAVDESS": ["audio", "video"],
    "SAVEE": ["audio"],
    "TESS": ["audio"],
    "IEMOCAP": ["audio", "video"],
    "FER2013": ["face"],
    "AffectNet": ["face"],
    "CK_PLUS": ["face"],
    "RAF_DB": ["face"],
    "GoEmotions": ["text"],
})

if "GoEmotions" not in ACTIVE_EMOTION_DATASETS:
    ACTIVE_EMOTION_DATASETS.append("GoEmotions")
if "GoEmotions" in ARCHIVED_DATASETS:
    ARCHIVED_DATASETS.remove("GoEmotions")
if "SAMM" not in ARCHIVED_DATASETS:
    ARCHIVED_DATASETS.append("SAMM")


def _looks_like_iemocap(root: Path) -> bool:
    """Return True only when a directory looks like an actual IEMOCAP release."""
    if not root.is_dir():
        return False
    try:
        names = {p.name.lower() for p in root.iterdir()}
    except Exception:
        return False

    # Normal release: Session1 ... Session5 at this level.
    session_count = sum(
        1 for n in names
        if n.startswith("session") and n[7:].isdigit()
    )
    if session_count >= 2:
        return True

    # Some extractions place the sessions one level below the release root.
    try:
        children = [p for p in root.iterdir() if p.is_dir()]
        nested_session_count = sum(
            1 for c in children
            if c.name.lower().startswith("session")
            and c.name[7:].isdigit()
        )
        return nested_session_count >= 2
    except Exception:
        return False


def resolve_iemocap_root(configured_path: Path) -> Path:
    """
    Resolve IEMOCAP robustly across common Kaggle/extracted layouts.

    Search order:
      1. configured path
      2. its existing ancestors/subdirectories
      3. PROJECT_DIR/datasets recursively
      4. PROJECT_DIR recursively as a final fallback

    The resolver does NOT silently choose an arbitrary folder named 'iemocap';
    it requires recognizable Session1/Session2/... structure.
    """
    configured_path = Path(configured_path)

    candidates = []

    # 1) Exact configured path and its descendants, if present.
    if configured_path.exists():
        candidates.append(configured_path)
        candidates.extend(
            p for p in configured_path.rglob("*")
            if p.is_dir()
        )

    # 2) Search the dataset area for IEMOCAP-like directories.
    search_roots = []
    datasets_root = Path(PROJECT_DIR) / "datasets"
    if datasets_root.exists():
        search_roots.append(datasets_root)
    if Path(PROJECT_DIR).exists():
        search_roots.append(Path(PROJECT_DIR))

    seen = set()
    for base in search_roots:
        try:
            for p in base.rglob("*"):
                if not p.is_dir():
                    continue
                key = str(p).lower()
                if key in seen:
                    continue
                seen.add(key)

                lname = p.name.lower()
                if (
                    "iemocap" in lname
                    or lname in {"session1", "session2", "session3", "session4", "session5"}
                ):
                    candidates.append(p)
        except Exception:
            pass

    # 3) Validate candidates by actual session structure.
    for c in candidates:
        try:
            c = c.resolve()
        except Exception:
            pass
        if _looks_like_iemocap(c):
            return c

    # Return the configured path unchanged if no valid release was found.
    return configured_path


# ---- Resolve IEMOCAP BEFORE inventory ----
DATASET_PATHS["IEMOCAP"] = resolve_iemocap_root(
    Path(DATASET_PATHS["IEMOCAP"])
)


def classify_physical_file(path: Path):
    ext = path.suffix.lower()
    if ext in AUDIO_EXTS:
        return "audio"
    if ext in VIDEO_EXTS:
        return "video"
    if ext in IMAGE_EXTS:
        return "face"
    if ext in TEXT_EXTS:
        return "text"
    return "other"


def is_declared_text_dataset(dataset_name: str) -> bool:
    return "text" in DATASET_MODALITIES.get(dataset_name, [])


def inventory_dataset(dataset_name: str):
    root = Path(DATASET_PATHS.get(dataset_name, ""))
    result = {
        "status": "NOT FOUND",
        "total_files": 0,
        "audio": 0,
        "video": 0,
        "face": 0,
        "text": 0,
        "other": 0,
        "metadata": 0,
    }

    if not root.exists():
        return result

    files = [p for p in root.rglob("*") if p.is_file()]
    result["status"] = "FOUND"
    result["total_files"] = len(files)

    for p in files:
        kind = classify_physical_file(p)

        if kind == "text" and not is_declared_text_dataset(dataset_name):
            result["metadata"] += 1
            continue

        result[kind] += 1

    return result


INVENTORY_DATASETS = (
    list(ACTIVE_EMOTION_DATASETS)
    + [x for x in ACTIVE_SARCASM_DATASETS
       if x not in ACTIVE_EMOTION_DATASETS]
)

print("=" * 115)
print(
    f"{'Dataset':<24}{'Status':<12}{'Files':>8}"
    f"{'Audio':>10}{'Video':>10}{'Face':>10}"
    f"{'Text':>10}{'Meta':>10}"
)
print("=" * 115)

for ds in INVENTORY_DATASETS:
    r = inventory_dataset(ds)
    print(
        f"{ds:<24}{r['status']:<12}{r['total_files']:>8}"
        f"{r['audio']:>10}{r['video']:>10}{r['face']:>10}"
        f"{r['text']:>10}{r['metadata']:>10}"
    )

print("=" * 115)
print("Logical modality policy:")
print("  image files -> FACE")
print("  media-dataset CSV/TXT/JSON/JSONL -> METADATA, not TEXT")
print("  GoEmotions -> TEXT and row-based (file count != sample count)")
print("  RAVDESS/IEMOCAP -> AUDIO + VIDEO when both physical modalities exist")
print("  SAMM -> archived/optional; not required for temporal video augmentation")
print("=" * 115)

print("\nRESOLVED ROOTS")
for ds in ["IEMOCAP", "GoEmotions"]:
    print(f"{ds}: {Path(DATASET_PATHS[ds])}")

if not Path(DATASET_PATHS["IEMOCAP"]).exists():
    print(
        "\nWARNING: IEMOCAP could not be resolved from PROJECT_DIR/datasets. "
        "The notebook will NOT pretend it is present."
    )
else:
    print("\nPASS: IEMOCAP release root resolved.")


Dataset                 Status         Files     Audio     Video      Face      Text      Meta
CREMA_D                 NOT FOUND          0         0         0         0         0         0
RAVDESS                 NOT FOUND          0         0         0         0         0         0
SAVEE                   NOT FOUND          0         0         0         0         0         0
TESS                    NOT FOUND          0         0         0         0         0         0
IEMOCAP                 NOT FOUND          0         0         0         0         0         0
FER2013                 NOT FOUND          0         0         0         0         0         0
AffectNet               NOT FOUND          0         0         0         0         0         0
CK_PLUS                 NOT FOUND          0         0         0         0         0         0
RAF_DB                  NOT FOUND          0         0         0         0         0         0
GoEmotions              NOT FOUND          0      

In [7]:
# ==========================================================
# PROJECT DIRECTORIES
# ==========================================================

# Initialization creates the project structure required by later stages.
# It does NOT read or validate future metadata/split/training artifacts.
from pathlib import Path

PROJECT_DIR = Path(r"C:\New folder\New EmoDect")

DIRECTORIES = {
    "base": PROJECT_DIR,
    "config": PROJECT_DIR / "config",
    "reports": PROJECT_DIR / "reports",
    "metadata": PROJECT_DIR / "cleaned_metadata",
    "splits": PROJECT_DIR / "cleaned_metadata" / "splits",
    "processed": PROJECT_DIR / "processed",
    "embeddings": PROJECT_DIR / "embeddings",
    "embeddings_today": PROJECT_DIR / "embeddings_today",
    "checkpoints": PROJECT_DIR / "checkpoints",
    "logs": PROJECT_DIR / "logs",
    "outputs": PROJECT_DIR / "outputs",
}

for directory in DIRECTORIES.values():
    directory.mkdir(parents=True, exist_ok=True)


# ==========================================================
# PROJECT CONFIGURATION
# ==========================================================

project_config = {
    "project_name": "Emotion Detection System V2",
    "seed": SEED,
    "device": str(DEVICE),
    "device_type": DEVICE_TYPE,
    "gpu_name": GPU_NAME,

    "num_classes": NUM_CLASSES,
    "emotion_labels": EMOTION_LABELS,
    "label2idx": LABEL2IDX,
    "idx2label": IDX2LABEL,
    "label_normalization": LABEL_NORMALIZATION,

    "modalities": MODALITIES,
    "modality2idx": MODALITY2IDX,
    "tasks": TASKS,

    "active_emotion_datasets": ACTIVE_EMOTION_DATASETS,
    "active_sarcasm_datasets": ACTIVE_SARCASM_DATASETS,
    "archived_datasets": ARCHIVED_DATASETS,

    "dataset_modalities": DATASET_MODALITIES,

    "audio_extensions": sorted(AUDIO_EXTENSIONS),
    "video_extensions": sorted(VIDEO_EXTENSIONS),
    "image_extensions": sorted(IMAGE_EXTENSIONS),
    "text_extensions": sorted(TEXT_EXTENSIONS),

    "directories": {
        name: str(path)
        for name, path in DIRECTORIES.items()
    },

    "dataset_paths": {
        name: str(path)
        for name, path in DATASET_PATHS.items()
    },
}


# ==========================================================
# SAVE CONFIGURATION
# ==========================================================

CONFIG_FILE_PATH = DIRECTORIES["config"] / "config.yaml"

with open(CONFIG_FILE_PATH, "w", encoding="utf-8") as yaml_file:
    yaml.dump(
        project_config,
        yaml_file,
        default_flow_style=False,
        sort_keys=False,
        allow_unicode=True,
    )


print("=" * 60)
print("PROJECT INITIALIZATION COMPLETE")
print("=" * 60)
print(f"Configuration saved to:\n{CONFIG_FILE_PATH}")
print(f"Processed data path:\n{DIRECTORIES['processed']}")
print("=" * 60)

print(
    "\nInitialization boundary:"
    "\n  This notebook configures the project and inventories raw datasets."
    "\n  It does NOT require cleaned metadata, split manifests, preprocessing,"
    "\n  embeddings, or training outputs to exist."
)


PROJECT INITIALIZATION COMPLETE
Configuration saved to:
C:\New folder\New EmoDect\config\config.yaml
Processed data path:
C:\New folder\New EmoDect\processed

Initialization boundary:
  This notebook configures the project and inventories raw datasets.
  It does NOT require cleaned metadata, split manifests, preprocessing,
  embeddings, or training outputs to exist.


In [8]:
# ==========================================================
# FINAL INITIALIZATION GATE
# ==========================================================
# This gate verifies configuration and GoEmotions presence.
# IEMOCAP is required by the emotion pipeline, so an unresolved IEMOCAP
# root blocks downstream verification rather than allowing a partial run.

print("\nDOWNSTREAM READINESS")
print(f"GoEmotions: {Path(DATASET_PATHS['GoEmotions']).exists()} -> {Path(DATASET_PATHS['GoEmotions'])}")
print(f"IEMOCAP:    {Path(DATASET_PATHS['IEMOCAP']).exists()} -> {Path(DATASET_PATHS['IEMOCAP'])}")

assert "GoEmotions" in ACTIVE_EMOTION_DATASETS
assert "GoEmotions" not in ARCHIVED_DATASETS
assert DATASET_MODALITIES["GoEmotions"] == ["text"]
assert "SAMM" in ARCHIVED_DATASETS

if not Path(DATASET_PATHS["GoEmotions"]).exists():
    raise FileNotFoundError(
        "GoEmotions root is missing. Restore the dataset before verification."
    )

if not Path(DATASET_PATHS["IEMOCAP"]).exists():
    raise FileNotFoundError(
        "IEMOCAP could not be resolved. Check the physical IEMOCAP extraction "
        "under the project datasets directory before downstream verification."
    )

print("PASS: GoEmotions active and IEMOCAP resolved.")
print("PASS: Initialization is ready for DATA VERIFICATION.")



DOWNSTREAM READINESS
GoEmotions: False -> D:\Emotion Detection system V2\datasets\debarshichanda\goemotions\versions\6\data\full_dataset
IEMOCAP:    False -> D:\Emotion Detection system V2\datasets\dejolilandry\iemocapfullrelease\versions\1\IEMOCAP_fullrelease


FileNotFoundError: GoEmotions root is missing. Restore the dataset before verification.